# Baseline


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import umap.umap_ as umap
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import pairwise_distances

In [ ]:
# Загрузка эмбеддингов
embeddings = np.load("embeddings_smart.npy")

# Стандартизация
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

In [ ]:
# 3 KMeans кластеризация

# Определение оптимального k
print("Определение оптимального k для KMeans...")
silhouette_scores = []
k_range = range(19, 20)

for k in tqdm(k_range, desc="Оценка k"):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(embeddings_scaled)
    if len(set(labels)) < 2:
        silhouette_scores.append(-1)
    else:
        silhouette_scores.append(silhouette_score(embeddings_scaled, labels))

# Визуализация метода локтя
plt.figure(figsize=(10, 5))
plt.plot(k_range, silhouette_scores, marker='o')
plt.title("Метод локтя для KMeans")
plt.xlabel("Число кластеров (k)")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.savefig("elbow_method_kmeans.png", dpi=200, bbox_inches='tight')
plt.close()

# Оптимальное k (по максимуму silhouette)
optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"\nОптимальное число кластеров (k): {optimal_k}")

# Кластеризация KMeans с оптимальным k
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
labels = kmeans.fit_predict(embeddings_scaled)

# Сохранение результатов
df["kmeans_cluster"] = labels
df.to_csv("articles_without_orcid_kmeans.csv", index=False)
print("Результаты KMeans сохранены в articles_without_orcid_kmeans.csv")

In [ ]:
# Кластеризация KMeans с оптимальным k
kmeans = KMeans(n_clusters=20, random_state=42)
labels = kmeans.fit_predict(embeddings_scaled)

# Сохранение результатов
df["kmeans_cluster"] = labels
df.to_csv("articles_without_orcid_kmeans.csv", index=False)
print("Результаты KMeans сохранены в articles_without_orcid_kmeans.csv")

# Experiment: using ilc topics embeddings as centroids for clusters

In [66]:
ARTIFACTS_PATH = "../data/artifacts/"

In [67]:
DEMO_PATH = "../demo_artifacts/"

In [68]:
SOURCE_DATA = "../data/source_data/"

In [69]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

# ЗАГРУЗКА ДАННЫХ 
embeddings_articles = np.load(ARTIFACTS_PATH + "embeddings_smart.npy")
print(f"Загружено эмбеддингов статей: {embeddings_articles.shape}")

df = pd.read_csv(SOURCE_DATA + ARTICLES_CSV)
print(f"Загружено статей из CSV: {len(df)}")

ilc_embeddings = np.load(ARTIFACTS_PATH + "ilc_root_embeddings.npy")
print(f"Загружено эмбеддингов ILC: {ilc_embeddings.shape}")

# Проверка совместимости размерностей
assert embeddings_articles.shape[1] == ilc_embeddings.shape[1]

# СТАНДАРТИЗАЦИЯ 
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_articles)
ilc_centroids_scaled = scaler.transform(ilc_embeddings)

# КЛАСТЕРИЗАЦИЯ (ПРИСВОЕНИЕ БЛИЖАЙШЕГО ЦЕНТРА ILC) 
n_clusters = len(ilc_centroids_scaled)
print(f"Число кластеров (корневых классов ILC): {n_clusters}")

kmeans = KMeans(
    n_clusters=n_clusters,
    init=ilc_centroids_scaled,
    n_init=1,
    random_state=42,
    max_iter=1          # без пересчёта центров
)
labels = kmeans.fit_predict(embeddings_scaled)

#  ДОБАВЛЕНИЕ ИНФОРМАЦИИ О КЛАССАХ ILC В DATAFRAME 
df["ilc_class_label"] = labels

# Расстояние до своего центра (косинусное)
distances = pairwise_distances(embeddings_scaled, ilc_centroids_scaled, metric="cosine")
df["min_distance_to_ilc_center"] = distances.min(axis=1)

# Загрузка метаинформации о классах ILC
ilc_info = pd.read_pickle(ARTIFACTS_PATH + "ilc_root_embeddings.pkl")
df["ilc_notation"] = df["ilc_class_label"].map(lambda i: ilc_info.iloc[i]["notation"])
df["ilc_label"] = df["ilc_class_label"].map(lambda i: ilc_info.iloc[i]["label"])

#  5. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ 
output_csv = ARTIFACTS_PATH + "articles_with_ilc_classes.csv"
df.to_csv(output_csv, index=False)
print(f"Результаты сохранены в {output_csv}")

Загружено эмбеддингов статей: (124282, 768)
Загружено статей из CSV: 124282
Загружено эмбеддингов ILC: (25, 768)
Число кластеров (корневых классов ILC): 25
Результаты сохранены в ../data/artifacts/articles_with_ilc_classes.csv


In [70]:
import numpy as np
import pandas as pd
import umap.umap_ as umap
import plotly.express as px
from sklearn.preprocessing import StandardScaler

INPUT_CSV = "articles_with_ilc_classes.csv"

#  ЗАГРУЗКА ДАННЫХ 
df = pd.read_csv(ARTIFACTS_PATH + INPUT_CSV)
print(f"Загружено статей: {len(df)}")

embeddings_articles = np.load(ARTIFACTS_PATH + "embeddings_smart.npy")
print(f"Загружено эмбеддингов статей: {embeddings_articles.shape}")

# СТАНДАРТИЗАЦИЯ
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_articles)

#  UMAP ПРОЕКЦИЯ 
print("Построение UMAP проекции...")
reducer = umap.UMAP(random_state=42)
embeddings_2d = reducer.fit_transform(embeddings_scaled)

df["umap_x"] = embeddings_2d[:, 0]
df["umap_y"] = embeddings_2d[:, 1]

#  4. ИНТЕРАКТИВНЫЙ ГРАФИК 
print("Создание интерактивного графика...")
# Для читаемой подписи классов
df["ilc_display_name"] = df["ilc_notation"] + " " + df["ilc_label"]

fig = px.scatter(
    df,
    x="umap_x",
    y="umap_y",
    color="ilc_display_name",
    hover_name="title",
    hover_data={
        "ilc_notation": True,
        "ilc_label": True,
        "min_distance_to_ilc_center": ":.3f",
        "umap_x": False,
        "umap_y": False
    },
    title="Распределение статей по корневым классам ILC (UMAP проекция)",
    color_discrete_sequence=px.colors.qualitative.Light24,
    labels={
        "ilc_notation": "Код ILC",
        "ilc_label": "Класс ILC",
        "min_distance_to_ilc_center": "Расст. до центра"
    }
)
fig.update_traces(marker=dict(size=6, opacity=0.7))

#  5. СОХРАНЕНИЕ HTML 
output_html = DEMO_PATH + "ilc_classification_umap.html"
fig.write_html(output_html)
print(f"Интерактивный график сохранён в {output_html}")

Загружено статей: 124282
Загружено эмбеддингов статей: (124282, 768)
Построение UMAP проекции...


d:\MIT-collect\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Создание интерактивного графика...
Интерактивный график сохранён в ../demo_artifacts/ilc_classification_umap.html


In [71]:
import os
import re
import pandas as pd
import numpy as np
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Константы для парсинга
MEANINGLESS_NER_CATEGORIES = {"DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "CARDINAL", "ORDINAL"}

def clean_word(word):
    """Очищает слово от возможного 'nan' и пустых строк."""
    if not word or word.lower() == 'nan':
        return None
    return word

def parse_ner_string(ner_str):
    if pd.isna(ner_str) or not isinstance(ner_str, str):
        return []
    words = []
    for entity in ner_str.split("; "):
        try:
            word, label, _ = entity.split("|")
            if label not in MEANINGLESS_NER_CATEGORIES:
                cleaned = clean_word(word.lower())
                if cleaned:
                    words.append(cleaned)
        except ValueError:
            continue
    return words

def parse_keywords_string(kw_str):
    if pd.isna(kw_str) or not isinstance(kw_str, str):
        return []
    words = []
    for kw in kw_str.split("; "):
        try:
            word, _ = kw.split("|")
            cleaned = clean_word(word.lower())
            if cleaned:
                words.append(cleaned)
        except ValueError:
            continue
    return words

def generate_wordcloud(text, title, output_path):
    """Генерирует и сохраняет облако слов."""
    if not text or len(text.strip()) == 0:
        print(f"Пропуск {output_path}: нет текста для облака.")
        return
    wordcloud = WordCloud(
        width=1200,
        height=800,
        background_color='white',
        stopwords=STOPWORDS_RU,
        colormap='viridis',
        max_words=200,
        relative_scaling=0.5,
        random_state=42
    ).generate(text)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title, fontsize=20, pad=20)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

# Загрузка данных
df = pd.read_csv(ARTIFACTS_PATH + "articles_with_ilc_classes.csv")
print(f"Загружено статей: {len(df)}")

# Группировка по классам ILC
for (notation, label), group in df.groupby(['ilc_notation', 'ilc_label']):
    # Создание папки для класса
    class_dir = os.path.join(DEMO_PATH + "clouds_ilc/", f"{notation}_{label.replace(' ', '_').replace('/', '_')}")
    os.makedirs(class_dir, exist_ok=True)
    
    # Сбор слов
    ner_words = []
    kw_words = []
    for _, row in group.iterrows():
        ner_words.extend(parse_ner_string(row['ner_entities']))
        kw_words.extend(parse_keywords_string(row['keywords']))
    
    ner_text = " ".join(ner_words)
    kw_text = " ".join(kw_words)
    combined_text = ner_text + " " + kw_text
    
    class_title = f"{notation} – {label}"
    
    # Генерация облаков
    generate_wordcloud(
        ner_text,
        f"NER entities: {class_title}",
        os.path.join(class_dir, "ner_wordcloud.png")
    )
    generate_wordcloud(
        kw_text,
        f"Keywords: {class_title}",
        os.path.join(class_dir, "keywords_wordcloud.png")
    )
    generate_wordcloud(
        combined_text,
        f"NER + Keywords: {class_title}",
        os.path.join(class_dir, "combined_wordcloud.png")
    )
    
    print(f"Облака слов сохранены для {notation} ({label}) в {class_dir}")

Загружено статей: 124282
Облака слов сохранены для a (forms) в ../demo_artifacts/clouds_ilc/a_forms
Облака слов сохранены для b (spacetime) в ../demo_artifacts/clouds_ilc/b_spacetime
Облака слов сохранены для c (branes) в ../demo_artifacts/clouds_ilc/c_branes
Облака слов сохранены для d (energy) в ../demo_artifacts/clouds_ilc/d_energy
Облака слов сохранены для e (atoms) в ../demo_artifacts/clouds_ilc/e_atoms
Облака слов сохранены для f (molecules) в ../demo_artifacts/clouds_ilc/f_molecules
Облака слов сохранены для g (continuum bodies) в ../demo_artifacts/clouds_ilc/g_continuum_bodies
Облака слов сохранены для h (celestial bodies) в ../demo_artifacts/clouds_ilc/h_celestial_bodies
Облака слов сохранены для i (rocks) в ../demo_artifacts/clouds_ilc/i_rocks
Облака слов сохранены для j (land) в ../demo_artifacts/clouds_ilc/j_land
Облака слов сохранены для k (genes) в ../demo_artifacts/clouds_ilc/k_genes
Облака слов сохранены для l (bacteria) в ../demo_artifacts/clouds_ilc/l_bacteria
Облака 

# Without common words

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
import umap.umap_ as umap
import plotly.express as px
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

ROOT_DATA_ARTIFACTS = "../data/artifacts/"
DEMO_PATH = "../demo_artifacts_filtered/"
FILTERED_CSV = ROOT_DATA_ARTIFACTS + "articles_filtered_for_ilc.csv"

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

# Загрузка отфильтрованного датасета
df = pd.read_csv(FILTERED_CSV)
print(f"Загружено статей: {len(df)}")

# Загрузка эмбеддингов статей (уже сгенерированы из отфильтрованных текстов)
embeddings_articles = np.load(ROOT_DATA_ARTIFACTS + "embeddings_filtered.npy")
print(f"Эмбеддинги статей: {embeddings_articles.shape}")

# Загрузка эмбеддингов ILC
ilc_embeddings = np.load(ROOT_DATA_ARTIFACTS + "ilc_root_embeddings.npy")
print(f"Эмбеддинги ILC: {ilc_embeddings.shape}")

assert embeddings_articles.shape[1] == ilc_embeddings.shape[1]

# Стандартизация
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_articles)
ilc_centroids_scaled = scaler.transform(ilc_embeddings)

# KMeans с фиксированными центрами
n_clusters = len(ilc_centroids_scaled)
print(f"Число кластеров (классов ILC): {n_clusters}")

kmeans = KMeans(
    n_clusters=n_clusters,
    init=ilc_centroids_scaled,
    n_init=1,
    random_state=42,
    max_iter=1
)
labels = kmeans.fit_predict(embeddings_scaled)

# Добавление результатов в DataFrame
df["ilc_class_label"] = labels
distances = pairwise_distances(embeddings_scaled, ilc_centroids_scaled, metric="cosine")
df["min_distance_to_ilc_center"] = distances.min(axis=1)

# Метаинформация ILC
ilc_info = pd.read_pickle(ROOT_DATA_ARTIFACTS + "ilc_root_embeddings.pkl")
df["ilc_notation"] = df["ilc_class_label"].map(lambda i: ilc_info.iloc[i]["notation"])
df["ilc_label"] = df["ilc_class_label"].map(lambda i: ilc_info.iloc[i]["label"])

# Сохранение результатов кластеризации
output_csv = ROOT_DATA_ARTIFACTS + "articles_with_ilc_classes_filtered.csv"
df.to_csv(output_csv, index=False)
print(f"Результаты сохранены в {output_csv}")

Загружено статей: 124282
Эмбеддинги статей: (124282, 768)
Эмбеддинги ILC: (25, 768)
Число кластеров (классов ILC): 25
Результаты сохранены в ../data/artifacts/articles_with_ilc_classes_filtered.csv


In [9]:
import pandas as pd
import numpy as np
import umap.umap_ as umap
import plotly.express as px
from sklearn.preprocessing import StandardScaler
import os

# Загрузка данных с метками ILC
df = pd.read_csv(ROOT_DATA_ARTIFACTS + "articles_with_ilc_classes_filtered.csv")
print(f"Загружено статей: {len(df)}")

# Эмбеддинги (те же, что использовались для кластеризации)
embeddings_articles = np.load(ROOT_DATA_ARTIFACTS + "embeddings_filtered.npy")

# Стандартизация (повторяем для воспроизводимости)
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_articles)

# UMAP проекция
print("Построение UMAP проекции...")
reducer = umap.UMAP(random_state=42)
embeddings_2d = reducer.fit_transform(embeddings_scaled)

df["umap_x"] = embeddings_2d[:, 0]
df["umap_y"] = embeddings_2d[:, 1]

# Читаемое имя класса
df["ilc_display_name"] = df["ilc_notation"] + " " + df["ilc_label"]

# Интерактивный scatter plot
fig = px.scatter(
    df,
    x="umap_x",
    y="umap_y",
    color="ilc_display_name",
    hover_name="title",
    hover_data={
        "ilc_notation": True,
        "ilc_label": True,
        "min_distance_to_ilc_center": ":.3f",
        "umap_x": False,
        "umap_y": False
    },
    title="Распределение статей по классам ILC (отфильтрованные эмбеддинги)",
    color_discrete_sequence=px.colors.qualitative.Light24,
    labels={
        "ilc_notation": "Код ILC",
        "ilc_label": "Класс ILC",
        "min_distance_to_ilc_center": "Расст. до центра"
    }
)
fig.update_traces(marker=dict(size=6, opacity=0.7))

os.makedirs(DEMO_PATH, exist_ok=True)
output_html = DEMO_PATH + "ilc_classification_filtered_umap.html"
fig.write_html(output_html)
print(f"График сохранён в {output_html}")

Загружено статей: 124282
Построение UMAP проекции...


d:\MIT-collect\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\MIT-collect\venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


График сохранён в ../demo_artifacts_filtered/ilc_classification_filtered_umap.html


In [11]:
import pandas as pd
import numpy as np
import os
import pickle
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Загрузка отфильтрованного датасета
df = pd.read_csv(ROOT_DATA_ARTIFACTS + "articles_with_ilc_classes_filtered.csv")
print(f"Загружено статей: {len(df)}")

# Загрузка списка высокочастотных слов (DF > 80%)
with open(ROOT_DATA_ARTIFACTS + "high_df_words.pkl", "rb") as f:
    high_df_words = pickle.load(f)

# Константы
MEANINGLESS_NER_CATEGORIES = {"DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "CARDINAL", "ORDINAL"}

def clean_word(word):
    if not word or word.lower() == 'nan':
        return None
    return word

def parse_ner_string(ner_str):
    if pd.isna(ner_str) or not isinstance(ner_str, str):
        return []
    words = []
    for entity in ner_str.split("; "):
        try:
            word, label, _ = entity.split("|")
            if label not in MEANINGLESS_NER_CATEGORIES:
                cleaned = clean_word(word.lower())
                if cleaned and cleaned not in high_df_words:
                    words.append(cleaned)
        except ValueError:
            continue
    return words

def parse_keywords_string(kw_str):
    if pd.isna(kw_str) or not isinstance(kw_str, str):
        return []
    words = []
    for kw in kw_str.split("; "):
        try:
            word, _ = kw.split("|")
            cleaned = clean_word(word.lower())
            if cleaned and cleaned not in high_df_words:
                words.append(cleaned)
        except ValueError:
            continue
    return words

def generate_wordcloud(text, title, output_path):
    if not text or len(text.strip()) == 0:
        print(f"Пропуск {output_path}: нет текста для облака.")
        return
    wordcloud = WordCloud(
        width=1200,
        height=800,
        background_color='white',
        colormap='viridis',
        max_words=200,
        relative_scaling=0.5,
        random_state=42
    ).generate(text)
    plt.figure(figsize=(12, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title, fontsize=20, pad=20)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

os.makedirs(DEMO_PATH, exist_ok=True)

# Группировка по классам ILC
for (notation, label), group in df.groupby(['ilc_notation', 'ilc_label']):
    class_dir = os.path.join(DEMO_PATH, f"{notation}_{label.replace(' ', '_').replace('/', '_')}")
    os.makedirs(class_dir, exist_ok=True)
    
    ner_words = []
    kw_words = []
    for _, row in group.iterrows():
        ner_words.extend(parse_ner_string(row['ner_entities']))
        kw_words.extend(parse_keywords_string(row['keywords']))
    
    ner_text = " ".join(ner_words)
    kw_text = " ".join(kw_words)
    combined_text = ner_text + " " + kw_text
    
    class_title = f"{notation} – {label}"
    
    generate_wordcloud(
        ner_text,
        f"NER entities: {class_title}",
        os.path.join(class_dir, "ner_wordcloud.png")
    )
    generate_wordcloud(
        kw_text,
        f"Keywords: {class_title}",
        os.path.join(class_dir, "keywords_wordcloud.png")
    )
    generate_wordcloud(
        combined_text,
        f"NER + Keywords: {class_title}",
        os.path.join(class_dir, "combined_wordcloud.png")
    )
    
    print(f"Облака слов сохранены для {notation} ({label}) в {class_dir}")

Загружено статей: 124282
Облака слов сохранены для a (forms) в ../demo_artifacts_filtered/a_forms
Облака слов сохранены для b (spacetime) в ../demo_artifacts_filtered/b_spacetime
Облака слов сохранены для c (branes) в ../demo_artifacts_filtered/c_branes
Облака слов сохранены для d (energy) в ../demo_artifacts_filtered/d_energy
Облака слов сохранены для e (atoms) в ../demo_artifacts_filtered/e_atoms
Облака слов сохранены для f (molecules) в ../demo_artifacts_filtered/f_molecules
Облака слов сохранены для g (continuum bodies) в ../demo_artifacts_filtered/g_continuum_bodies
Облака слов сохранены для h (celestial bodies) в ../demo_artifacts_filtered/h_celestial_bodies
Облака слов сохранены для i (rocks) в ../demo_artifacts_filtered/i_rocks
Облака слов сохранены для j (land) в ../demo_artifacts_filtered/j_land
Облака слов сохранены для k (genes) в ../demo_artifacts_filtered/k_genes
Облака слов сохранены для l (bacteria) в ../demo_artifacts_filtered/l_bacteria
Облака слов сохранены для m (or